# **Notebook 3 - Basket Analysis**

Afonso Fernandes 20241710, Lourenço Lima 20241711, Lucas Casimiro 20241796

## Imports

In [10]:
import os
import sys
import warnings
from pathlib import Path

def _find_project_root(start, marker="requirements.txt"):
    path = Path(start).resolve()
    for candidate in [path] + list(path.parents):
        if (candidate / marker).exists():
            return str(candidate)
    raise RuntimeError(f"Could not find project root (marker={marker!r}, searched from {start})")

PROJECT_ROOT = _find_project_root(os.path.abspath("."))
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

warnings.filterwarnings("ignore")

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [49]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import importlib
from functions.basket import *

from functions.preprocessing import FEATURE_COLS
from functions.basket import *
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# **1. Merging data**

In [50]:
customer_basket = pd.read_csv('data/customer_basket.csv')
customer = pd.read_csv('data/ci_clustered.csv')

In [51]:
merged_data = pd.merge(customer_basket, customer, on="customer_id", how="inner")
merged_data

,invoice_id,list_of_goods,customer_id,customer_name,customer_gender,kids_home,teens_home,number_complaints,distinct_stores_visited,lifetime_spend_groceries,...,share_nonalcohol_drinks,share_alcohol_drinks,share_meat,share_fish,share_hygiene,share_videogames,share_petfood,dietary_preference,final_cluster_nr,final_cluster_label
0,3700630,"['chicken', 'rice', 'pepper', 'whole wheat ric...",12912,Msc. George Davis,male,1.0,0.0,2.0,3.0,13092.0,...,0.020800,0.020462,0.052198,0.032525,0.036359,0.009977,0.028129,omnivore,4,Gamers
1,10242376,"['low fat yogurt', 'tomatoes', 'pepper', 'aspa...",22853,Patricia Silva,female,1.0,1.0,1.0,6.0,15518.0,...,0.007247,0.004986,0.074887,0.014905,0.019531,0.001902,0.019377,omnivore,4,Gamers
2,91550,"['cake', 'tomatoes', 'pancakes', 'iPad', 'fina...",19,Phd. Evelyn Quinn,female,1.0,2.0,3.0,3.0,8264.0,...,0.026508,0.034148,0.068031,0.093343,0.054411,0.011161,0.035411,omnivore,0,Tech enthusiasts
3,3137503,"['cereals', 'megaman zero', 'final fantasy XIX...",10995,Msc. David Stupar,male,3.0,4.0,0.0,2.0,24573.0,...,0.028657,0.061240,0.049489,0.041370,0.028016,0.010897,0.001415,omnivore,2,Average customer
4,7165061,"['rice', 'frozen smoothie', 'black tea', 'tea'...",27807,Noel Froelich,female,6.0,1.0,1.0,2.0,26093.0,...,0.025166,0.036475,0.042300,0.032471,0.052298,0.006285,0.011989,omnivore,2,Average customer
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98699,7685930,"['yams', 'airpods', 'shrimp']",21357,Frank Erdmann,male,1.0,0.0,0.0,8.0,15106.0,...,0.013355,0.038252,0.032097,0.023358,0.032536,0.000824,0.012916,omnivore,4,Gamers
98700,2998960,"['cake', 'chocolate bread', 'cologne', 'beer',...",7817,Marian Mitchell,female,0.0,0.0,1.0,2.0,13164.0,...,0.014037,0.027575,0.021447,0.033921,0.008504,0.061652,0.007409,omnivore,1,Clean and healthy
98701,4409376,"['chicken', 'frozen smoothie', 'milk', 'body s...",35497,Judith Roane,female,0.0,1.0,0.0,2.0,6439.0,...,0.050386,0.012645,0.098263,0.035811,0.044015,0.004633,0.040927,omnivore,0,Tech enthusiasts
98702,3650163,"['metroid fusion', 'cat food', 'melons', 'ring...",13412,Phd. Mark Lunde,male,1.0,1.0,0.0,2.0,17407.0,...,0.021467,0.020131,0.005159,0.000967,0.015110,0.000000,0.010135,omnivore,4,Gamers


# **2. Dataframe by cluster**

In [52]:
cluster_dataframes = {
    cluster: group
    for cluster, group in merged_data.groupby("final_cluster_label", sort=False)}


**NOTE**: Here is the code where we ran after running the association rules the first time and seeing unrelated results to the vegan cluster. We copied the output to show why we needed to separate the cluster and redo the clustering process. The output was originated from checking the `dietary_preference` column distribution in the vegan cluster.

--- Vegans ---

dietary_preference

omnivore       3615

vegetarian     3019

carnivore         1

pescatarian       1

Name: count, dtype: int64

## Parameter by cluster

In [60]:
params_by_cluster = {
        "Bargain hunters": {"min_support": 0.01, "min_threshold": 1, "min_confidence": 0.10},

        "Tech enthusiasts": {"min_support": 0.02, "min_threshold": 1.5, "min_confidence": 0.1},

        "Big families (big spenders)": {"min_support": 0.01, "min_threshold": 1.5, "min_confidence": 0.1},

        "Clean and healthy": {"min_support": 0.01, "min_threshold": 1, "min_confidence": 0.10},
        
        "Average customer": {"min_support": 0.03, "min_threshold": 1, "min_confidence": 0.10},

        "Gamers": {"min_support": 0.03, "min_threshold": 1.5, "min_confidence": 0.1},
        
        "Loyal big spenders": {"min_support": 0.02, "min_threshold": 1, "min_confidence": 0.10},

        "Karens": {"min_support": 0.08, "min_threshold": 1.5, "min_confidence": 0.1},

        "Vegans": {"min_support": 0.05, "min_threshold": 1, "min_confidence": 0.1}}

# **3. Association rules**

Note: some baskets were lost due to outlier removal, we decided to keep it with less baskets to guarantee previous clean results.

In [61]:
generate_rules_for_all_clusters = generate_rules_for_all_clusters
generate_association_rules = generate_association_rules

In [62]:
all_cluster_rules = generate_rules_for_all_clusters(
    cluster_dataframes=cluster_dataframes,
    params_by_cluster=params_by_cluster
)

for cluster_name, rules in all_cluster_rules.items():
    print(f"CLUSTER: {cluster_name}")

    if rules.empty:
        print("No rules found.")
    else:
        display(rules.head(30))

CLUSTER: Gamers


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,frozenset({bluetooth headphones}),frozenset({energy drink}),0.126229,0.124297,0.038199,0.302618,2.434634,1.0,0.022509,1.255701,0.674388,0.179907,0.203632,0.304969
1,frozenset({energy drink}),frozenset({bluetooth headphones}),0.124297,0.126229,0.038199,0.307320,2.434634,1.0,0.022509,1.261437,0.672900,0.179907,0.207253,0.304969
2,frozenset({airpods}),frozenset({bluetooth headphones}),0.210824,0.126229,0.059316,0.281352,2.228907,1.0,0.032704,1.215854,0.698640,0.213568,0.177533,0.375630
3,frozenset({bluetooth headphones}),frozenset({airpods}),0.126229,0.210824,0.059316,0.469908,2.228907,1.0,0.032704,1.488753,0.631000,0.213568,0.328297,0.375630
4,frozenset({energy drink}),frozenset({airpods}),0.124297,0.210824,0.058028,0.466851,2.214405,1.0,0.031823,1.480215,0.626253,0.209418,0.324423,0.371048
5,frozenset({airpods}),frozenset({energy drink}),0.210824,0.124297,0.058028,0.275244,2.214405,1.0,0.031823,1.208273,0.694917,0.209418,0.172373,0.371048
6,frozenset({pancakes}),frozenset({airpods}),0.079660,0.210824,0.031589,0.396552,1.880957,1.0,0.014795,1.307777,0.508894,0.122016,0.235343,0.273194
7,frozenset({airpods}),frozenset({pancakes}),0.210824,0.079660,0.031589,0.149837,1.880957,1.0,0.014795,1.082545,0.593475,0.122016,0.076251,0.273194
8,frozenset({energy bar}),frozenset({airpods}),0.090261,0.210824,0.035753,0.396101,1.878818,1.0,0.016723,1.306800,0.514159,0.134746,0.234772,0.282843
9,frozenset({airpods}),frozenset({energy bar}),0.210824,0.090261,0.035753,0.169585,1.878818,1.0,0.016723,1.095522,0.592708,0.134746,0.087194,0.282843


CLUSTER: Tech enthusiasts


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,frozenset({energy drink}),frozenset({airpods}),0.081975,0.158616,0.030474,0.371747,2.343686,1.0,0.017471,1.339244,0.624516,0.145033,0.253310,0.281935
1,frozenset({airpods}),frozenset({energy drink}),0.158616,0.081975,0.030474,0.192123,2.343686,1.0,0.017471,1.136343,0.681403,0.145033,0.119984,0.281935
2,frozenset({bluetooth headphones}),frozenset({airpods}),0.087003,0.158616,0.031388,0.360771,2.274483,1.0,0.017588,1.316247,0.613737,0.146515,0.240264,0.279329
3,frozenset({airpods}),frozenset({bluetooth headphones}),0.158616,0.087003,0.031388,0.197887,2.274483,1.0,0.017588,1.138240,0.665974,0.146515,0.121450,0.279329
4,frozenset({pancakes}),frozenset({airpods}),0.068719,0.158616,0.020722,0.301552,1.901140,1.0,0.009822,1.204648,0.508976,0.100295,0.169882,0.216098
5,frozenset({airpods}),frozenset({pancakes}),0.158616,0.068719,0.020722,0.130644,1.901140,1.0,0.009822,1.071231,0.563358,0.100295,0.066494,0.216098
6,frozenset({iphone 10}),frozenset({airpods}),0.071918,0.158616,0.021484,0.298729,1.883340,1.0,0.010077,1.199798,0.505374,0.102770,0.166526,0.217088
7,frozenset({airpods}),frozenset({iphone 10}),0.158616,0.071918,0.021484,0.135447,1.883340,1.0,0.010077,1.073481,0.557449,0.102770,0.068451,0.217088
8,frozenset({airpods}),frozenset({laptop}),0.158616,0.090507,0.023465,0.147935,1.634504,1.0,0.009109,1.067398,0.461375,0.103984,0.063142,0.203597
9,frozenset({laptop}),frozenset({airpods}),0.090507,0.158616,0.023465,0.259259,1.634504,1.0,0.009109,1.135868,0.426824,0.103984,0.119616,0.203597


CLUSTER: Average customer


...

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,frozenset({airpods}),frozenset({energy drink}),0.075826,0.058612,0.011047,0.145686,2.485592,1.0,0.006602,1.101922,0.646719,0.089526,0.092495,0.167079
1,frozenset({energy drink}),frozenset({airpods}),0.058612,0.075826,0.011047,0.188472,2.485592,1.0,0.006602,1.138808,0.634894,0.089526,0.121889,0.167079
2,frozenset({airpods}),frozenset({bluetooth headphones}),0.075826,0.058719,0.010779,0.142150,2.420833,1.0,0.006326,1.097255,0.635074,0.087088,0.088635,0.162856
3,frozenset({bluetooth headphones}),frozenset({airpods}),0.058719,0.075826,0.010779,0.183562,2.420833,1.0,0.006326,1.131958,0.623532,0.087088,0.116575,0.162856
4,frozenset({napkins}),"frozenset({toilet paper, dog food})",0.185221,0.037913,0.014532,0.078460,2.069473,1.0,0.007510,1.043999,0.634264,0.069666,0.042145,0.230885
5,"frozenset({toilet paper, dog food})",frozenset({napkins}),0.037913,0.185221,0.014532,0.383310,2.069473,1.0,0.007510,1.321213,0.537150,0.069666,0.243120,0.230885
6,frozenset({napkins}),"frozenset({chicken, dog food})",0.185221,0.036304,0.013889,0.074986,2.065480,1.0,0.007165,1.041817,0.633118,0.066890,0.040139,0.228778
7,"frozenset({chicken, dog food})",frozenset({napkins}),0.036304,0.185221,0.013889,0.382570,2.065480,1.0,0.007165,1.319630,0.535284,0.066890,0.242212,0.228778
8,frozenset({napkins}),"frozenset({fresh bread, cooking oil})",0.185221,0.031478,0.011958,0.064563,2.051052,1.0,0.006128,1.035368,0.628938,0.058408,0.034160,0.222230
9,"frozenset({fresh bread, cooking oil})",frozenset({napkins}),0.031478,0.185221,0.011958,0.379898,2.051052,1.0,0.006128,1.313943,0.529100,0.058408,0.238932,0.222230


CLUSTER: Loyal big spenders


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,frozenset({cereals}),frozenset({eggs}),0.108562,0.106645,0.028562,0.263096,2.467017,1.0,0.016985,1.212308,0.667071,0.153030,0.175127,0.265460
1,frozenset({eggs}),frozenset({cereals}),0.106645,0.108562,0.028562,0.267825,2.467017,1.0,0.016985,1.217520,0.665640,0.153030,0.178658,0.265460
2,frozenset({butter}),frozenset({eggs}),0.111629,0.106645,0.028498,0.255295,2.393867,1.0,0.016594,1.199608,0.655431,0.150168,0.166395,0.261260
3,frozenset({eggs}),frozenset({butter}),0.106645,0.111629,0.028498,0.267226,2.393867,1.0,0.016594,1.212339,0.651775,0.150168,0.175148,0.261260
4,frozenset({butter}),frozenset({cereals}),0.111629,0.108562,0.028051,0.251288,2.314689,1.0,0.015932,1.190628,0.639346,0.145993,0.160107,0.254838
5,frozenset({cereals}),frozenset({butter}),0.108562,0.111629,0.028051,0.258387,2.314689,1.0,0.015932,1.197890,0.637147,0.145993,0.165199,0.254838
6,frozenset({cereals}),frozenset({fresh bread}),0.108562,0.121981,0.028115,0.258976,2.123087,1.0,0.014872,1.184873,0.593410,0.138889,0.156027,0.244732
7,frozenset({fresh bread}),frozenset({cereals}),0.121981,0.108562,0.028115,0.230487,2.123087,1.0,0.014872,1.158444,0.602478,0.138889,0.136773,0.244732
8,frozenset({fresh bread}),frozenset({eggs}),0.121981,0.106645,0.027540,0.225773,2.117041,1.0,0.014531,1.153866,0.600947,0.136956,0.133348,0.242006
9,frozenset({eggs}),frozenset({fresh bread}),0.106645,0.121981,0.027540,0.258238,2.117041,1.0,0.014531,1.183695,0.590631,0.136956,0.155188,0.242006


CLUSTER: Karens


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,frozenset({shampoo}),frozenset({deodorant}),0.269613,0.281603,0.126756,0.470140,1.669511,1.0,0.050832,1.355823,0.549054,0.298628,0.262441,0.460131
1,frozenset({deodorant}),frozenset({shampoo}),0.281603,0.269613,0.126756,0.450122,1.669511,1.0,0.050832,1.328270,0.558218,0.298628,0.247141,0.460131
2,frozenset({shower gel}),frozenset({shampoo}),0.269613,0.269613,0.119219,0.442186,1.640076,1.0,0.046528,1.309373,0.534336,0.283850,0.236276,0.442186
3,frozenset({shampoo}),frozenset({shower gel}),0.269613,0.269613,0.119219,0.442186,1.640076,1.0,0.046528,1.309373,0.534336,0.283850,0.236276,0.442186
4,frozenset({shower gel}),frozenset({deodorant}),0.269613,0.281603,0.124358,0.461245,1.637926,1.0,0.048434,1.333439,0.533240,0.291332,0.250060,0.451426
5,frozenset({deodorant}),frozenset({shower gel}),0.281603,0.269613,0.124358,0.441606,1.637926,1.0,0.048434,1.308014,0.542140,0.291332,0.235482,0.451426
6,frozenset({deodorant}),frozenset({toothpaste}),0.281603,0.273724,0.124358,0.441606,1.613326,1.0,0.047276,1.300651,0.529182,0.288553,0.231154,0.447962
7,frozenset({toothpaste}),frozenset({deodorant}),0.273724,0.281603,0.124358,0.454318,1.613326,1.0,0.047276,1.316511,0.523441,0.288553,0.240417,0.447962
8,frozenset({shower gel}),frozenset({tooth brush}),0.269613,0.267215,0.116136,0.430750,1.611998,1.0,0.044091,1.287281,0.519795,0.276059,0.223169,0.432683
9,frozenset({tooth brush}),frozenset({shower gel}),0.267215,0.269613,0.116136,0.434615,1.611998,1.0,0.044091,1.291841,0.518094,0.276059,0.225911,0.432683


NOTE: after much debugging, we couldn't find the cause root to the label switch between the 'Karens' and 'Clean and healthy' clusters. To fix the promotions to be as intended in the app, we've considered them to be swapped, despite acknowledging the output above is incorrect.

# **4. Promotions**

In the app, you can find promotion in the the "Promotions and Basket" section.